## 1. Configuración Inicial de Parámetros

Ejecuta la siguiente celda para configurar los parámetros de auditoría. 
Todos los valores están documentados con comentarios inline para facilitar inspección antes del envío.

In [2]:
from pathlib import Path
from auditoria_config import AuditConfig

# ============================================================================
# PARÁMETROS DE AUDITORÍA - Configurar antes de ejecutar
# ============================================================================

# Ruta del PDF a auditar (relativa o absoluta)
# Ejemplo: "../plan_tesis_cables_pinn.pdf" o "C:/ruta/al/documento.pdf"
INPUT_PDF = Path("../plan_tesis_cables_pinn.pdf")

# Directorio de salida para reportes, artefactos JSON y manifests
# Se crea si no existe
OUTPUT_DIR = Path("./auditoria_resultados")

# Configuración de API y Docker
API_URL = "http://127.0.0.1:8000"  # URL del servicio sloptotal local
CONTAINER_NAME = "sloptotal"  # Nombre del contenedor Docker
START_CONTAINER = True  # Iniciar contenedor si está detenido
STARTUP_TIMEOUT = 60  # Segundos de espera para que el contenedor responda

# Configuración de modo de análisis
# Opciones: 'whole' (global), 'chunks' (fragmentos), 'both' (ambos, default)
ANALYSIS_MODE = "both"

# Configuración de fragmentación (solo usado si ANALYSIS_MODE incluye 'chunks')
# Máximo de caracteres por fragmento para chunking
MAX_CHARS = 5000

# Configuración de umbrales y timeouts
REVIEW_THRESHOLD = 65.0  # Puntuación mínima para priorizar lectura humana
REQUEST_TIMEOUT = 120  # Segundos de timeout para requests HTTP

# Modo seco: True no envía requests ni inicia Docker (solo valida config)
# Usar para inspeccionar sin efectos colaterales
DRY_RUN = False

# Máximo de fragmentos a procesar; None procesa todos
MAX_CHUNKS = None

# Detener auditoría después de N timeouts consecutivos
STOP_AFTER_TIMEOUTS = 3

# ============================================================================
# Instanciar configuración
# ============================================================================
config = AuditConfig(
    input_path=INPUT_PDF,
    output_dir=OUTPUT_DIR,
    api_url=API_URL,
    container_name=CONTAINER_NAME,
    start_container=START_CONTAINER,
    startup_timeout=STARTUP_TIMEOUT,
    mode=ANALYSIS_MODE,
    max_chars=MAX_CHARS,
    review_threshold=REVIEW_THRESHOLD,
    request_timeout=REQUEST_TIMEOUT,
    dry_run=DRY_RUN,
    max_chunks=MAX_CHUNKS,
    stop_after_timeouts=STOP_AFTER_TIMEOUTS,
)

print("✓ Configuración cargada exitosamente")
print(f"  - Entrada: {config.input_path}")
print(f"  - Salida: {config.output_dir}")
print(f"  - Modo: {config.mode} | DRY_RUN: {config.dry_run}")


✓ Configuración cargada exitosamente
  - Entrada: C:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\plan_tesis_cables_pinn.pdf
  - Salida: C:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA\auditoria_resultados
  - Modo: both | DRY_RUN: False


## 2. Importaciones y Verificación de Entorno

In [3]:
import json
import sys
from typing import Any

# Importar funciones de backend
from auditoria_config import prepare_document, diagnose_service, run_audit

print("✓ Importaciones completadas")
print(f"  - Python {sys.version.split()[0]}")
print(f"  - Directorio de trabajo: {Path.cwd()}")
print(f"  - Módulo backend: auditoria_config.py disponible")


✓ Importaciones completadas
  - Python 3.12.9
  - Directorio de trabajo: c:\usr\ths_mia_fiis\tesis_MIA_lk\plan_v2\auditoria_IA
  - Módulo backend: auditoria_config.py disponible


## 3. Diagnóstico de Servicio

Verifica el estado de Docker y la API sloptotal. En `DRY_RUN=True`, este paso devuelve None sin efectos colaterales.

In [4]:
if config.dry_run:
    print("⊘ MODO SECO (DRY_RUN=True)")
    print("  No se enviarán requests ni se iniciará Docker")
    diagnostics = None
else:
    print("Diagnosticando servicio...")
    diagnostics = diagnose_service(config)
    
    if diagnostics:
        docker = diagnostics.get("docker", {})
        health = diagnostics.get("health")
        
        print(f"\n✓ Estado Docker:")
        print(f"  - Disponible: {docker.get('available', False)}")
        print(f"  - Existe contenedor '{config.container_name}': {docker.get('exists', False)}")
        print(f"  - En ejecución: {docker.get('running', False)}")
        
        if health:
            print(f"\n✓ Salud API:")
            print(f"  - Status: {health.get('status', 'desconocido')}")
            engines = health.get("engines", {})
            print(f"  - Motores cargados: {engines.get('loaded', 'N/D')}/{engines.get('total', 'N/D')}")
            failed = engines.get("failed", [])
            if failed:
                print(f"  - Motores fallidos: {', '.join(failed)}")
        else:
            print(f"\n⚠ No se pudo conectar a la API: {docker.get('health_error', 'sin detalles')}")
    else:
        print("⚠ Diagnóstico no disponible")


Diagnosticando servicio...

✓ Estado Docker:
  - Disponible: True
  - Existe contenedor 'sloptotal': True
  - En ejecución: True

✓ Salud API:
  - Status: healthy
  - Motores cargados: 12/23
  - Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate

✓ Estado Docker:
  - Disponible: True
  - Existe contenedor 'sloptotal': True
  - En ejecución: True

✓ Salud API:
  - Status: healthy
  - Motores cargados: 12/23
  - Motores fallidos: perplexity, cross_perplexity, classifier_openai, classifier_chatgpt, classifier_remodetect, classifier_fakespot, classifier_e5, classifier_tmr, classifier_bert_raid, classifier_desklib, classifier_superannotate


## 4. Extracción y Vista Previa del Documento

In [5]:
print("Extrayendo documento...")
document, chunks = prepare_document(config)

metadata = document.metadata
print(f"\n✓ Extracción completada:")
print(f"  - Total de páginas en PDF: {metadata.get('pdf_pages', 'N/D')}")
print(f"  - Cuerpo detectado: páginas {metadata.get('body_page_start', 'N/D')}–{metadata.get('body_page_end', 'N/D')}")
print(f"  - Palabras en cuerpo: {metadata.get('body_words', 0):,}")
print(f"  - Caracteres en cuerpo: {metadata.get('body_characters', 0):,}")
print(f"  - Fragmentos generados: {len(chunks)}")
if config.max_chunks and len(chunks) > config.max_chunks:
    print(f"  - Limitado a {config.max_chunks} fragmentos (max_chunks)")

print(f"\nPrimer fragmento (primeros 500 caracteres):")
if chunks:
    first_chunk = chunks[0]
    excerpt = first_chunk.text[:500]
    if len(first_chunk.text) > 500:
        excerpt += "…"
    print(f"  ID: {first_chunk.chunk_id}")
    print(f"  Página: {first_chunk.page_start}")
    print(f"  Capítulo: {first_chunk.chapter or '(inicio)'}")
    print(f"  Sección: {first_chunk.section or '(sin sección)'}")
    print(f"\n{excerpt}")
else:
    print("  No hay fragmentos para mostrar")


Extrayendo documento...

✓ Extracción completada:
  - Total de páginas en PDF: 73
  - Cuerpo detectado: páginas 11–60
  - Palabras en cuerpo: 17,787
  - Caracteres en cuerpo: 119,150
  - Fragmentos generados: 50

Primer fragmento (primeros 500 caracteres):
  ID: p011_c001
  Página: 11
  Capítulo: CAPÍTULO 1. PLANTEAMIENTO DEL PROBLEMA
  Sección: 1.1 INTRODUCCIÓN

CAPÍTULO 1. PLANTEAMIENTO DEL PROBLEMA
1.1 INTRODUCCIÓN
En el Perú, entre 2010 y 2025, la máxima demanda del Sistema Eléctrico Interconectado
Nacional (SEIN) aumentó de 4579 MW a 7942 MW. En el mismo período, la participación
conjunta de la generación solar y eólica pasó de un valor cercano a cero a 9,5 %. Estos
cambios aumentan la exigencia de transportar energía y aprovechar la capacidad disponible
de la red (Ministerio de Energía y Minas, 2025; Ministerio de Energía y Minas, 2026b; COES,
2026…


## 5. Ejecución de Auditoría

Se enviarán requests a la API según los parámetros configurados. 
El progreso se mostrará en tiempo real.

In [ ]:
print("Iniciando auditoría...")
print(f"Modo: {config.mode} | Modo de fragmentación: {config.chunk_mode}")

try:
    report_path = run_audit(config, document, chunks, diagnostics)
    print(f"\n✓ Auditoría completada exitosamente")
    print(f"Reporte disponible en: {report_path}")
    
    # Mostrar tamaño del directorio de salida
    output_files = list(config.output_dir.glob("*"))
    print(f"\nArtefactos generados ({len(output_files)} archivo(s)):")
    for f in sorted(output_files)[:10]:
        size_bytes = f.stat().st_size if f.is_file() else 0
        size_kb = size_bytes / 1024
        print(f"  - {f.name} ({size_kb:.1f} KB)")
    if len(output_files) > 10:
        print(f"  ... y {len(output_files) - 10} archivo(s) más")
        
except Exception as e:
    print(f"\n✗ Error durante auditoría:")
    print(f"  {type(e).__name__}: {str(e)}")
    report_path = None


Iniciando auditoría...
Modo: both | Modo de fragmentación: structure


## 6. Visualización de Reporte

Se muestra el reporte Markdown completo con resultados y análisis.

In [ ]:
if report_path and report_path.is_file():
    report_content = report_path.read_text(encoding="utf-8")
    # Mostrar como Markdown en el notebook
    from IPython.display import Markdown, display
    display(Markdown(report_content))
else:
    print("⚠ No se pudo cargar el reporte. Verifica que la auditoría haya completado exitosamente.")


# Auditoría Sloptotal: Detección de IA en Documentos

**Versión**: 1.0  
**Última actualización**: 2026-11-13  
**Descripción**: Flujo reproducible de auditoría de documentos con sloptotal local.

Este cuaderno automatiza la evaluación de documentos contra un servicio de detección de IA usando la API local de sloptotal. Todas las configuraciones están centralizadas al inicio para facilitar la inspección antes del envío.

## Características

- ✓ Soporte para PDF con extracción de cuerpo de documento
- ✓ Extracción preservando texto completo (portada, referencias, bibliografía se excluyen)
- ✓ Segmentación por caracteres para control granular
- ✓ Diagnóstico Docker e inicio de contenedor existente
- ✓ Modo seco (dry_run=True) para inspeccionar antes de enviar
- ✓ Puntuación global y por fragmentos secuencialmente en `/api/analyze`
- ✓ Reportes aislados por ejecución con entrada exacta y artefactos (json, manifest)

## Límites de interpretación

- Una puntuación de detector no establece quién escribió un pasaje
- La puntuación global y las puntuaciones por fragmento responden a entradas distintas
- Los comentarios heurísticos solo señalan carga sintáctica y repetición de conectores
- Para interpretar una alerta, contrastarla con historial de versiones y decisiones de redacción